# Methionine Excision Analysis Across ProteoBench Submissions

**Related issue:** [#504 - To explore: parameter methionine excision during search](https://github.com/Proteobench/ProteoBench/issues/504)

Some search engines remove the initiator methionine from the N-terminus of protein sequences,
because N-terminal methionine excision (NME) is a common co-translational modification.

This notebook analyzes all publicly submitted ProteoBench datasets to determine:
1. Which submissions contain peptides starting with methionine at position 1 (i.e., NME was **not** applied)?
2. Which submissions appear to have removed initiator methionines (NME **was** applied)?
3. Is there a systematic difference between software tools?

**Data source:** https://proteobench.cubimed.rub.de/datasets/

In [2]:
!pip install blinker

  Obtaining dependency information for blinker from https://files.pythonhosted.org/packages/10/cb/f2ad4230dc2eb1a74edf38f1a38b9b52277f75bef262d8908e60d957e13c/blinker-1.9.0-py3-none-any.whl.metadata
Using cached blinker-1.9.0-py3-none-any.whl (8.5 kB)


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
streamlit 1.39.0 requires pillow<11,>=7.1.0, but you have pillow 11.3.0 which is incompatible.


In [3]:
import glob
import io
import os
import zipfile
from collections import defaultdict

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import requests
from bs4 import BeautifulSoup
from tqdm.notebook import tqdm

from proteobench.utils.server_io import get_merged_json

## 1. Load all submission metadata

We load the JSON results from all modules to get the mapping between `intermediate_hash`, `software_name`, and other metadata.

In [4]:
# Load results from all ion-level modules
repos = {
    "DDA_ion": "https://github.com/Proteobench/Results_quant_ion_DDA/archive/refs/heads/main.zip",
    "DDA_Astral_ion": "https://github.com/Proteobench/Results_quant_ion_DDA_Astral/archive/refs/heads/main.zip",
    "DIA_AIF_ion": "https://github.com/Proteobench/Results_quant_ion_DIA/archive/refs/heads/main.zip",
    "DIA_diaPASEF_ion": "https://github.com/Proteobench/Results_quant_ion_DIA_diaPASEF/archive/refs/heads/main.zip",
    "DIA_Astral_ion": "https://github.com/Proteobench/Results_quant_ion_DIA_Astral/archive/refs/heads/main.zip",
    "DIA_ZenoTOF_ion": "https://github.com/Proteobench/Results_quant_ion_DIA_ZenoTOF/archive/refs/heads/main.zip",
}

all_metadata = []
for module_name, repo_url in repos.items():
    try:
        df = get_merged_json(repo_url, write_to_file=False)
        df["module"] = module_name
        all_metadata.append(df)
        print(f"{module_name}: {len(df)} datapoints")
    except Exception as e:
        print(f"{module_name}: FAILED - {e}")

metadata = pd.concat(all_metadata, ignore_index=True)
print(f"\nTotal: {len(metadata)} datapoints across {metadata['module'].nunique()} modules")
print(f"Unique hashes: {metadata['intermediate_hash'].nunique()}")
print(f"Software tools: {sorted(metadata['software_name'].unique())}")

Combined 47 JSON files into 'combined_results.json'.
DDA_ion: 47 datapoints
Combined 7 JSON files into 'combined_results.json'.
DDA_Astral_ion: 7 datapoints
Combined 0 JSON files into 'combined_results.json'.
DIA_AIF_ion: 0 datapoints
Combined 26 JSON files into 'combined_results.json'.
DIA_diaPASEF_ion: 26 datapoints
Combined 42 JSON files into 'combined_results.json'.
DIA_Astral_ion: 42 datapoints
Combined 4 JSON files into 'combined_results.json'.
DIA_ZenoTOF_ion: 4 datapoints

Total: 126 datapoints across 5 modules
Unique hashes: 126
Software tools: ['AlphaDIA', 'AlphaPept', 'DIA-NN', 'FragPipe', 'FragPipe (DIA-NN quant)', 'MSAngel', 'MaxQuant', 'PEAKS', 'ProlineStudio', 'Sage', 'Spectronaut', 'WOMBAT', 'i2MassChroQ', 'quantms']


## 2. Download performance data from the server

Each dataset folder on `proteobench.cubimed.rub.de/datasets/` contains a zip with `result_performance.csv`.
The `precursor ion` column contains sequences in ProForma notation (e.g., `AAAM[Oxidation]PEPTIDEK/2`).

We extract the bare peptide sequence and check for N-terminal methionine patterns.

In [5]:
DATASETS_BASE_URL = "https://proteobench.cubimed.rub.de/datasets/"

def get_available_hashes():
    """Get all dataset hashes available on the server."""
    resp = requests.get(DATASETS_BASE_URL, timeout=30)
    resp.raise_for_status()
    soup = BeautifulSoup(resp.text, "html.parser")
    return [
        a["href"].strip("/")
        for a in soup.find_all("a")
        if a["href"].endswith("/") and a["href"] != "../"
    ]

server_hashes = get_available_hashes()
print(f"Datasets available on server: {len(server_hashes)}")

# Find which metadata entries have data on the server
metadata_with_data = metadata[metadata["intermediate_hash"].isin(server_hashes)]
print(f"Metadata entries with server data: {len(metadata_with_data)}")

Datasets available on server: 212
Metadata entries with server data: 126


In [6]:
import re

def extract_bare_sequence(precursor_ion: str) -> str:
    """Extract the bare amino acid sequence from a ProForma precursor ion string.
    
    Examples:
        'AAAM[Oxidation]PEPTIDEK/2' -> 'AAAMEPTIDEK'  (charge removed, but keep sequence)
        'M[Acetyl]-PEPTIDEK/3'      -> 'MPEPTIDEK'
        '[Acetyl]-MPEPTIDEK/2'      -> 'MPEPTIDEK'
    """
    if not isinstance(precursor_ion, str):
        return ""
    # Remove charge state (e.g., /2, |Z=2)
    seq = re.split(r'[/|]', precursor_ion)[0]
    # Remove modifications in brackets
    seq = re.sub(r'\[.*?\]', '', seq)
    # Remove leading/trailing dashes (from N-term/C-term mods)
    seq = seq.strip('-')
    # Keep only uppercase letters (amino acids)
    seq = ''.join(c for c in seq if c.isupper())
    return seq

# Test
test_cases = [
    "AAAAAAALQAK/2",
    "M[Oxidation]AAAPEPTIDEK/3",
    "[Acetyl]-MPEPTIDEK/2",
    "MPEPTIDEK|Z=2",
    "AAAM[Oxidation]PEPTIDEK/2",
]
for tc in test_cases:
    print(f"  {tc:40s} -> {extract_bare_sequence(tc)}")

  AAAAAAALQAK/2                            -> AAAAAAALQAK
  M[Oxidation]AAAPEPTIDEK/3                -> MAAAPEPTIDEK
  [Acetyl]-MPEPTIDEK/2                     -> MPEPTIDEK
  MPEPTIDEK|Z=2                            -> MPEPTIDEK
  AAAM[Oxidation]PEPTIDEK/2                -> AAAMPEPTIDEK


In [7]:
def load_performance_data(intermediate_hash: str) -> pd.DataFrame:
    """Download and extract result_performance.csv from the server."""
    folder_url = f"{DATASETS_BASE_URL}{intermediate_hash}/"
    
    # Find the zip file
    resp = requests.get(folder_url, timeout=15)
    resp.raise_for_status()
    soup = BeautifulSoup(resp.text, "html.parser")
    zip_files = [a["href"] for a in soup.find_all("a") if a["href"].endswith(".zip")]
    
    if not zip_files:
        return None
    
    # Download the zip
    zip_url = f"{folder_url}{zip_files[0]}"
    zip_resp = requests.get(zip_url, timeout=60)
    zip_resp.raise_for_status()
    
    # Extract result_performance.csv
    with zipfile.ZipFile(io.BytesIO(zip_resp.content)) as z:
        if "result_performance.csv" in z.namelist():
            with z.open("result_performance.csv") as f:
                return pd.read_csv(f)
    return None

In [8]:
def analyze_methionine_excision(perf_df: pd.DataFrame, precursor_col: str = "precursor ion") -> dict:
    """Analyze a single dataset for methionine excision patterns.
    
    We look at peptides that map to the N-terminus of proteins.
    Since we don't have protein position info in the intermediate file,
    we use a heuristic: check the proportion of peptides starting with M.
    
    If NME is applied, we expect FEWER peptides starting with M
    (because the M was cleaved and the peptide starts with the 2nd residue).
    
    We also look for [Acetyl] N-term modifications, which are common
    on the residue exposed after NME.
    """
    if precursor_col not in perf_df.columns:
        # Try alternative column names
        for alt in ["peptidoform", "proforma"]:
            if alt in perf_df.columns:
                precursor_col = alt
                break
        else:
            return None
    
    precursors = perf_df[precursor_col].dropna().unique()
    total = len(precursors)
    
    if total == 0:
        return None
    
    sequences = [extract_bare_sequence(p) for p in precursors]
    sequences = [s for s in sequences if len(s) > 0]
    
    # Count sequences starting with M
    starts_with_m = sum(1 for s in sequences if s.startswith("M"))
    pct_starts_m = starts_with_m / len(sequences) * 100 if sequences else 0
    
    # Count N-terminal acetylation (indicator of NME processing)
    has_nterm_acetyl = sum(
        1 for p in precursors
        if isinstance(p, str) and (
            p.startswith("[Acetyl]") or 
            p.startswith("[acetyl]") or
            "[Acetyl]-" in p[:15]
        )
    )
    pct_nterm_acetyl = has_nterm_acetyl / total * 100 if total else 0
    
    # Amino acid frequency at position 1
    first_aa_counts = defaultdict(int)
    for s in sequences:
        if s:
            first_aa_counts[s[0]] += 1
    
    return {
        "total_precursors": total,
        "unique_sequences": len(sequences),
        "starts_with_M": starts_with_m,
        "pct_starts_M": round(pct_starts_m, 2),
        "n_nterm_acetyl": has_nterm_acetyl,
        "pct_nterm_acetyl": round(pct_nterm_acetyl, 2),
        "first_aa_distribution": dict(first_aa_counts),
    }

## 3. Analyze all available datasets

This downloads each dataset's performance CSV and checks for methionine excision patterns.
This may take a few minutes depending on your connection.

In [9]:
results = []
errors = []

hashes_to_process = metadata_with_data["intermediate_hash"].unique()
print(f"Processing {len(hashes_to_process)} datasets...\n")

for h in tqdm(hashes_to_process):
    # Get metadata for this hash
    meta_row = metadata_with_data[metadata_with_data["intermediate_hash"] == h].iloc[0]
    
    try:
        perf_df = load_performance_data(h)
        if perf_df is None:
            errors.append({"hash": h, "error": "No performance data found"})
            continue
        
        analysis = analyze_methionine_excision(perf_df)
        if analysis is None:
            errors.append({"hash": h, "error": "No precursor column found"})
            continue
        
        results.append({
            "intermediate_hash": h,
            "software_name": meta_row.get("software_name", "unknown"),
            "software_version": meta_row.get("software_version", "unknown"),
            "module": meta_row.get("module", "unknown"),
            "id": meta_row.get("id", "unknown"),
            **analysis,
        })
    except Exception as e:
        errors.append({"hash": h, "error": str(e)[:100]})

print(f"\nSuccessfully analyzed: {len(results)}")
print(f"Errors: {len(errors)}")

results_df = pd.DataFrame(results)
results_df.head()

Processing 126 datasets...



  0%|          | 0/126 [00:00<?, ?it/s]


Successfully analyzed: 126
Errors: 0


,intermediate_hash,software_name,software_version,module,id,total_precursors,unique_sequences,starts_with_M,pct_starts_M,n_nterm_acetyl,pct_nterm_acetyl,first_aa_distribution
0,00e2f863939301a2a71178652972dad895b27520,MaxQuant,2.5.1.0,DDA_ion,MaxQuant_20250605_124520,49811,49811,1227,2.46,739,1.48,"{'A': 4774, 'C': 413, 'D': 3267, 'E': 4235, 'F..."
1,0280a06fabdbe84746419d0810deae56e7ab2406,MaxQuant,2.3.1.0,DDA_ion,MaxQuant_20250610_104137,49848,49848,1231,2.47,740,1.48,"{'A': 4778, 'C': 414, 'D': 3274, 'E': 4239, 'F..."
2,1438823a0441d7a32c0d1095074cbc4a187ce2fb,i2MassChroQ,1.0.16,DDA_ion,i2MassChroQ_20250610_104346,76957,76957,2144,2.79,0,0.00,"{'A': 7320, 'C': 1284, 'D': 4538, 'E': 6015, '..."
3,1bfa914c771321b285a9ca40d4aa538cb9fdc42e,AlphaPept,0.5.0,DDA_ion,AlphaPept_20250605_073532,59608,59608,1573,2.64,140,0.23,"{'A': 5286, 'C': 498, 'D': 3970, 'E': 5081, 'F..."
4,254d6c77ce656888918e738772ad5f5f6f1543e4,MaxQuant,1.5.8.2,DDA_ion,MaxQuant_20250610_104638,49184,49184,1201,2.44,732,1.49,"{'A': 4713, 'C': 412, 'D': 3241, 'E': 4170, 'F..."


## 4. Results: Methionine excision across tools

In [10]:
# Summary by software tool
if not results_df.empty:
    summary = results_df.groupby("software_name").agg(
        n_submissions=("id", "count"),
        mean_pct_starts_M=("pct_starts_M", "mean"),
        std_pct_starts_M=("pct_starts_M", "std"),
        min_pct_starts_M=("pct_starts_M", "min"),
        max_pct_starts_M=("pct_starts_M", "max"),
        mean_pct_nterm_acetyl=("pct_nterm_acetyl", "mean"),
    ).round(2).sort_values("mean_pct_starts_M", ascending=False)
    
    print("Percentage of precursors starting with Methionine by tool:")
    print("(Higher = more M-starting peptides = NME likely NOT applied or not relevant)")
    print()
    display(summary)
else:
    print("No results to display.")

Percentage of precursors starting with Methionine by tool:
(Higher = more M-starting peptides = NME likely NOT applied or not relevant)



,n_submissions,mean_pct_starts_M,std_pct_starts_M,min_pct_starts_M,max_pct_starts_M,mean_pct_nterm_acetyl
software_name,,,,,,
AlphaDIA,16,3.60,0.21,3.19,4.03,1.26
DIA-NN,30,3.28,1.14,2.80,7.57,0.96
Spectronaut,11,3.26,0.09,3.10,3.41,1.15
PEAKS,8,3.25,0.29,2.60,3.50,0.00
FragPipe (DIA-NN quant),7,3.18,0.11,3.03,3.28,1.18
WOMBAT,5,2.70,0.20,2.46,2.92,1.76
Sage,3,2.68,0.19,2.47,2.80,0.01
i2MassChroQ,13,2.68,0.17,2.35,2.98,0.00
AlphaPept,4,2.64,0.01,2.63,2.65,0.24


In [11]:
# Visualization: box plot of % M-starting peptides per tool
if not results_df.empty:
    fig = px.box(
        results_df,
        x="software_name",
        y="pct_starts_M",
        color="module",
        title="Percentage of Precursors Starting with Methionine by Software Tool",
        labels={
            "pct_starts_M": "% precursors starting with M",
            "software_name": "Software Tool",
            "module": "Module",
        },
        hover_data=["id", "software_version", "total_precursors"],
    )
    fig.update_layout(
        xaxis_tickangle=-45,
        height=500,
        showlegend=True,
    )
    fig.show()

In [12]:
# Visualization: scatter of % M-starting vs total precursors
if not results_df.empty:
    fig = px.scatter(
        results_df,
        x="unique_sequences",
        y="pct_starts_M",
        color="software_name",
        symbol="module",
        title="% M-Starting Precursors vs Total Unique Sequences",
        labels={
            "unique_sequences": "Unique Sequences",
            "pct_starts_M": "% starting with M",
            "software_name": "Software Tool",
        },
        hover_data=["id", "software_version"],
        size="total_precursors",
        size_max=15,
    )
    fig.update_layout(height=500)
    fig.show()

In [13]:
# Visualization: N-terminal amino acid distribution across tools
if not results_df.empty:
    # Aggregate first AA distributions
    aa_data = []
    for _, row in results_df.iterrows():
        dist = row["first_aa_distribution"]
        total = sum(dist.values())
        for aa, count in dist.items():
            aa_data.append({
                "software_name": row["software_name"],
                "amino_acid": aa,
                "percentage": count / total * 100,
            })
    
    aa_df = pd.DataFrame(aa_data)
    aa_summary = aa_df.groupby(["software_name", "amino_acid"])["percentage"].mean().reset_index()
    
    # Focus on key amino acids: M (methionine), A (alanine - common after NME), S (serine)
    key_aas = ["M", "A", "S", "T", "V", "G", "L", "K"]
    aa_filtered = aa_summary[aa_summary["amino_acid"].isin(key_aas)]
    
    fig = px.bar(
        aa_filtered,
        x="software_name",
        y="percentage",
        color="amino_acid",
        barmode="group",
        title="N-terminal Amino Acid Distribution by Tool (avg across submissions)",
        labels={
            "percentage": "% of sequences",
            "software_name": "Software Tool",
            "amino_acid": "First AA",
        },
    )
    fig.update_layout(xaxis_tickangle=-45, height=500)
    fig.show()

## 5. Detailed comparison: Same module, different tools

Let's compare tools within the same module to control for dataset effects.

In [14]:
if not results_df.empty:
    for module in results_df["module"].unique():
        module_df = results_df[results_df["module"] == module]
        if len(module_df) < 2:
            continue
        
        print(f"\n{'='*60}")
        print(f"Module: {module} ({len(module_df)} submissions)")
        print(f"{'='*60}")
        
        module_summary = module_df.groupby("software_name").agg(
            n=("id", "count"),
            mean_pct_M=("pct_starts_M", "mean"),
            mean_nterm_acetyl=("pct_nterm_acetyl", "mean"),
            mean_precursors=("total_precursors", "mean"),
        ).round(2).sort_values("mean_pct_M", ascending=False)
        
        display(module_summary)


Module: DDA_ion (47 submissions)


,n,mean_pct_M,mean_nterm_acetyl,mean_precursors
software_name,,,,
WOMBAT,5,2.70,1.76,48156.60
AlphaPept,4,2.64,0.24,55984.50
Sage,2,2.64,0.00,58742.50
i2MassChroQ,10,2.64,0.00,71204.70
MSAngel,1,2.62,1.32,65553.00
PEAKS,1,2.60,0.00,74435.00
ProlineStudio,1,2.56,1.44,59062.00
FragPipe,7,2.54,1.28,68493.14
MaxQuant,15,2.43,1.43,49924.40



Module: DDA_Astral_ion (7 submissions)


,n,mean_pct_M,mean_nterm_acetyl,mean_precursors
software_name,,,,
FragPipe,2,2.92,1.40,40963.50
MaxQuant,1,2.86,1.66,23869.00
i2MassChroQ,3,2.83,0.00,36453.33
Sage,1,2.78,0.04,30351.00



Module: DIA_diaPASEF_ion (26 submissions)


,n,mean_pct_M,mean_nterm_acetyl,mean_precursors
software_name,,,,
AlphaDIA,6,3.75,1.21,86355.00
DIA-NN,8,3.49,1.09,113182.62
FragPipe (DIA-NN quant),4,3.27,1.21,86696.75
PEAKS,3,3.26,0.00,107209.33
Spectronaut,5,3.18,1.07,127455.00



Module: DIA_Astral_ion (42 submissions)


,n,mean_pct_M,mean_nterm_acetyl,mean_precursors
software_name,,,,
AlphaDIA,8,3.48,1.20,78086.12
PEAKS,3,3.46,0.00,139950.00
Spectronaut,6,3.32,1.21,116012.83
DIA-NN,21,3.22,0.91,123778.52
FragPipe (DIA-NN quant),3,3.06,1.14,93855.33
MaxQuant,1,2.93,1.78,23890.00



Module: DIA_ZenoTOF_ion (4 submissions)


,n,mean_pct_M,mean_nterm_acetyl,mean_precursors
software_name,,,,
AlphaDIA,2,3.58,1.64,65613.5
PEAKS,1,3.24,0.00,132330.0
DIA-NN,1,3.00,1.08,103911.0


## 6. Statistical test: Is there a significant difference between tools?

We use a Kruskal-Wallis test (non-parametric) to check if the percentage of M-starting
peptides differs significantly between software tools.

In [15]:
from scipy import stats

if not results_df.empty and results_df["software_name"].nunique() > 1:
    groups = [group["pct_starts_M"].values for _, group in results_df.groupby("software_name") if len(group) >= 2]
    
    if len(groups) >= 2:
        stat, p_value = stats.kruskal(*groups)
        print(f"Kruskal-Wallis test: H={stat:.2f}, p={p_value:.4e}")
        if p_value < 0.05:
            print("=> Significant difference between tools (p < 0.05)")
            print("   This suggests methionine excision handling varies across tools.")
        else:
            print("=> No significant difference between tools (p >= 0.05)")
            print("   Tools handle methionine excision similarly.")
    else:
        print("Not enough groups with >= 2 submissions for statistical test.")

Kruskal-Wallis test: H=98.37, p=1.1559e-16
=> Significant difference between tools (p < 0.05)
   This suggests methionine excision handling varies across tools.


## 7. Conclusions & Recommendations

Based on this analysis:

1. **If tools show similar % M-starting peptides:** NME handling is consistent, and it may not need to be a tracked parameter.

2. **If tools differ significantly:** NME should be added as a parsed parameter in ProteoBench, and users should be informed during submission.

3. **If N-terminal acetylation patterns differ:** This could indicate that some tools are reporting NME-related modifications while others are not, which would affect quantification comparisons.

### Action items for issue #504:
- [ ] If heterogeneous: add "methionine excision" as a parameter in `ProteoBenchParameters`
- [ ] Update parameter JSON files (`quant_lfq_DDA_ion.json`, etc.)
- [ ] Add parsing logic for each tool's parameter file
- [ ] Consider flagging submissions where NME handling is ambiguous

In [16]:
# Save results for further analysis
if not results_df.empty:
    output_path = "temp_results/methionine_excision_results.csv"
    os.makedirs("temp_results", exist_ok=True)
    results_df.drop(columns=["first_aa_distribution"]).to_csv(output_path, index=False)
    print(f"Results saved to {output_path}")
    
    # Also save errors
    if errors:
        pd.DataFrame(errors).to_csv("temp_results/methionine_excision_errors.csv", index=False)
        print(f"Errors saved to temp_results/methionine_excision_errors.csv")

Results saved to temp_results/methionine_excision_results.csv
